# Recurrent 4×4 sudoku

This notebook constructs the tensor-network geometry proposed in [optyx#13](https://github.com/rel-int/optyx/issues/13): sixteen cell channels exchange port-addressed messages with four row, four column and four square channels. The same `CMap` has finite stream semantics through `unroll` and approximate stationary semantics through `fix`.

Each cell has three incoming message qubits, three outgoing message qubits and two prediction qubits. Each constraint has four incoming and four outgoing message qubits. The two directions are separate ports, so the recurrent network has 96 edges and 192 memory wires; the 32 unpaired prediction wires form its boundary.

In [ ]:
import numpy as np

from optyx.channel import Diagram, qubit
from optyx.core.backends import DiscopyBackend
from optyx.interaction import Box, CMap
from optyx.qubits import Ket

The identity channels below expose the wiring independently of a learning ansatz. Replacing them with parametrised eight-qubit channels changes the local computation without changing the `CMap`, which is the separation needed for a backend-neutral training experiment.

In [ ]:
size = 4
n_cells = size ** 2
n_constraints = 3 * size


def cell_box(index):
    ports = qubit ** 8
    return Box(
        f"cell_{index}", qubit ** 3, qubit ** 5, Diagram.id(ports))


def constraint_box(kind, index):
    ports = qubit ** 8
    return Box(
        f"{kind}_{index}", qubit ** 4, qubit ** 4, Diagram.id(ports))


cells = [cell_box(index) for index in range(n_cells)]
constraints = [
    constraint_box(kind, index)
    for kind in ("row", "column", "square")
    for index in range(size)
]
boxes = cells + constraints

In [ ]:
def memberships(row, column):
    square = 2 * (row // 2) + column // 2
    square_position = 2 * (row % 2) + column % 2
    return (
        (n_cells + row, column),
        (n_cells + size + column, row),
        (n_cells + 2 * size + square, square_position),
    )


edges = []
for cell in range(n_cells):
    row, column = divmod(cell, size)
    for slot, (constraint, position) in enumerate(
            memberships(row, column)):
        edges.append(((cell, slot), (constraint, size + position)))
        edges.append(((cell, 3 + slot), (constraint, position)))

sudoku = CMap(boxes, edges)
summary = {
    "boxes": len(sudoku.boxes),
    "edges": len(sudoku.edges),
    "prediction_wires": len(sudoku.boundary),
    "memory_wires": len(sudoku.memory),
}
assert summary == {
    "boxes": 28, "edges": 96,
    "prediction_wires": 32, "memory_wires": 192}
summary

## Finite message passing

`unroll(n_steps)` copies the local channels through time and routes every output message to its partner's input at the next step. Memory is deliberately left open here, so a caller can choose the initial messages and final effects before tensor contraction. Materialising all 192 memory wires is unnecessary for a documentation run, so the executable cell uses one row slice of the same construction.

In [ ]:
row_boxes = [cell_box(index) for index in range(size)] + [
    constraint_box("row", 0)]
row_edges = [
    edge
    for cell in range(size)
    for edge in (
        ((cell, 0), (size, size + cell)),
        ((cell, 3), (size, cell)),
    )
]
row_map = CMap(row_boxes, row_edges)
n_steps = 2
unrolled = row_map.unroll(n_steps)
unrolled_summary = {
    "steps": n_steps,
    "domain_wires": len(unrolled.dom),
    "codomain_wires": len(unrolled.cod),
    "diagram_boxes": len(unrolled.boxes),
}
assert unrolled_summary["domain_wires"] == 64
assert unrolled_summary["codomain_wires"] == 64
unrolled_summary

The prediction boundary contains four probabilities per cell. A training experiment can compare them with a solved grid using the mean squared distance below. Keeping this loss separate from the diagram makes the remaining batching and PyTorch-backed contraction work explicit.

In [ ]:
def prediction_loss(probabilities, solution):
    targets = np.eye(size)[np.asarray(solution) - 1]
    return np.mean((np.asarray(probabilities) - targets) ** 2)


solution = np.array([
    1, 2, 3, 4,
    3, 4, 1, 2,
    2, 1, 4, 3,
    4, 3, 2, 1,
])
uniform_predictions = np.full((n_cells, size), 1 / size)
assert np.isclose(prediction_loss(uniform_predictions, solution), 3 / 16)

## Approximate stationary semantics

`CMap.fix(input_state, initial_state, ...)` prepares the boundary input at every step, initializes the recurrent memory and delegates to `Diagram.fix`. The power method can refine depth and bond dimension; the eigen method is exact only when the memory transfer matrix fits in memory. The full sudoku has 192 memory qubits, so dense evaluation is intentionally not attempted in this executable example.

In [ ]:
wire = Box("wire", qubit, qubit ** 2, Diagram.id(qubit ** 3))
probe = CMap([wire], [((0, 1), (0, 2))])
fixed = probe.fix(
    Ket(1), Ket(0) @ Ket(0), n_steps=2,
    backend=DiscopyBackend())
assert np.allclose(fixed.density_matrix, [[0, 0], [0, 1]])

For the full experiment, the next implementation round replaces the identity channels with trainable local ansätze, contracts the unrolling with bounded bond dimension and threads batches through the backend. This differs from the classical CMap-GNN in [discopy#416](https://github.com/discopy/discopy/pull/416): both keep messages port-addressed, but this notebook compiles recurrent quantum channels to a tensor network rather than pooling classical feature vectors. Dynamic `n_steps` then becomes a convergence policy around `fix`, not a trainable integer hidden inside the diagram.